In [3]:
!pip install -U "huggingface_hub<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 11.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0


In [4]:
import huggingface_hub
from huggingface_hub import HfApi

print("Hugging Face instalado!")
print("Versão:", huggingface_hub.__version__)

api = HfApi()

print("API inicializada com sucesso!")

Hugging Face instalado!
Versão: 1.33.0
API inicializada com sucesso!


In [5]:
import huggingface_hub
from huggingface_hub import HfApi

print("Biblioteca:", huggingface_hub.__version__)

api = HfApi()

print("Conexão com a API inicializada!")

Biblioteca: 1.33.0
Conexão com a API inicializada!


In [6]:
!pip install -q transformers torch

In [7]:
from transformers import pipeline

print("Transformers importado com sucesso!")

Transformers importado com sucesso!


In [8]:
from transformers import pipeline

modelo = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

print("Modelo carregado com sucesso!")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Modelo carregado com sucesso!


In [9]:
avaliacoes = [
    "A comida chegou muito rápido e estava deliciosa!",
    "Meu pedido chegou atrasado e completamente frio.",
    "O entregador foi educado, mas a comida estava apenas razoável."
]

resultados = modelo(avaliacoes)

for avaliacao, resultado in zip(avaliacoes, resultados):
    print("Avaliação:", avaliacao)
    print("Sentimento:", resultado["label"])
    print("Confiança:", round(resultado["score"] * 100, 2), "%")
    print("-" * 60)

Avaliação: A comida chegou muito rápido e estava deliciosa!
Sentimento: positive
Confiança: 92.49 %
------------------------------------------------------------
Avaliação: Meu pedido chegou atrasado e completamente frio.
Sentimento: negative
Confiança: 87.9 %
------------------------------------------------------------
Avaliação: O entregador foi educado, mas a comida estava apenas razoável.
Sentimento: positive
Confiança: 55.43 %
------------------------------------------------------------


In [10]:
resultados_detalhados = modelo(avaliacoes, top_k=None)

for avaliacao, resultado in zip(avaliacoes, resultados_detalhados):
    print("\nAvaliação:", avaliacao)

    for classe in sorted(resultado, key=lambda x: x["score"], reverse=True):
        porcentagem = classe["score"] * 100

        print(f'{classe["label"]}: {porcentagem:.2f}%')


Avaliação: A comida chegou muito rápido e estava deliciosa!
positive: 92.49%
neutral: 6.12%
negative: 1.39%

Avaliação: Meu pedido chegou atrasado e completamente frio.
negative: 87.90%
neutral: 10.71%
positive: 1.39%

Avaliação: O entregador foi educado, mas a comida estava apenas razoável.
positive: 55.43%
neutral: 28.43%
negative: 16.14%


In [11]:
import pandas as pd

def analisar_avaliacao(texto):
    resultado = modelo(texto, top_k=None)

    # Organiza as classes retornadas pelo modelo
    if resultado and isinstance(resultado[0], list):
        resultado = resultado[0]

    melhor_resultado = max(
        resultado,
        key=lambda x: x["score"]
    )

    return {
        "avaliacao": texto,
        "sentimento": melhor_resultado["label"],
        "pontuacao": round(melhor_resultado["score"], 4)
    }


# Testando nossa função
avaliacoes_teste = [
    "A entrega foi excelente!",
    "Meu pedido chegou completamente errado.",
    "O atendimento foi razoável."
]

dados = [
    analisar_avaliacao(texto)
    for texto in avaliacoes_teste
]

df = pd.DataFrame(dados)

display(df)

,avaliacao,sentimento,pontuacao
0,A entrega foi excelente!,positive,0.9150
1,Meu pedido chegou completamente errado.,negative,0.8888
2,O atendimento foi razoável.,positive,0.8876


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

dataset = [
    ("A comida estava deliciosa!", "positive"),
    ("A entrega foi excelente!", "positive"),
    ("Adorei o atendimento!", "positive"),
    ("Tudo chegou perfeito e rápido!", "positive"),

    ("Meu pedido chegou completamente errado.", "negative"),
    ("A comida estava fria e horrível.", "negative"),
    ("O atendimento foi péssimo.", "negative"),
    ("Meu pedido foi cancelado sem explicação.", "negative"),

    ("O pedido chegou às 19 horas.", "neutral"),
    ("O restaurante confirmou meu pedido.", "neutral"),
    ("O pedido contém dois hambúrgueres.", "neutral"),
    ("A entrega ocorreu às 20h30.", "neutral")
]

df_teste = pd.DataFrame(dataset, columns=["avaliacao", "esperado"])

# Executa as previsões
df_teste["previsto"] = [
    analisar_avaliacao(texto)["sentimento"]
    for texto in df_teste["avaliacao"]
]

display(df_teste)

# Avalia o desempenho
print("\nRelatório de classificação:")
print(classification_report(
    df_teste["esperado"],
    df_teste["previsto"],
    labels=["negative", "neutral", "positive"],
    zero_division=0
))

print("Matriz de confusão:")
print(confusion_matrix(
    df_teste["esperado"],
    df_teste["previsto"],
    labels=["negative", "neutral", "positive"]
))

,avaliacao,esperado,previsto
0,A comida estava deliciosa!,positive,positive
1,A entrega foi excelente!,positive,positive
2,Adorei o atendimento!,positive,positive
3,Tudo chegou perfeito e rápido!,positive,positive
4,Meu pedido chegou completamente errado.,negative,negative
5,A comida estava fria e horrível.,negative,negative
6,O atendimento foi péssimo.,negative,negative
7,Meu pedido foi cancelado sem explicação.,negative,negative
8,O pedido chegou às 19 horas.,neutral,neutral
9,O restaurante confirmou meu pedido.,neutral,positive



Relatório de classificação:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         4
     neutral       1.00      0.75      0.86         4
    positive       0.80      1.00      0.89         4

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

Matriz de confusão:
[[4 0 0]
 [0 3 1]
 [0 0 4]]


In [13]:

import pandas as pd
from sklearn.model_selection import train_test_split

dataset = [
    # POSITIVE — 20 avaliações
    ("A comida estava deliciosa!", "positive"),
    ("A entrega foi excelente e muito rápida!", "positive"),
    ("Adorei o atendimento do restaurante!", "positive"),
    ("Meu pedido chegou perfeito!", "positive"),
    ("O entregador foi muito educado!", "positive"),
    ("A embalagem estava ótima e bem organizada!", "positive"),
    ("Comida quentinha e muito saborosa!", "positive"),
    ("Chegou antes do horário previsto!", "positive"),
    ("Excelente experiência, comprarei novamente!", "positive"),
    ("Tudo estava maravilhoso!", "positive"),
    ("O restaurante caprichou no meu pedido!", "positive"),
    ("Entrega rápida e atendimento impecável!", "positive"),
    ("O lanche estava muito gostoso!", "positive"),
    ("Gostei bastante da qualidade dos alimentos!", "positive"),
    ("Meu pedido veio completo e bem embalado!", "positive"),
    ("Muito satisfeito com o serviço!", "positive"),
    ("A comida estava fresca e saborosa!", "positive"),
    ("O entregador foi atencioso e gentil!", "positive"),
    ("Ótimo restaurante, recomendo!", "positive"),
    ("Tudo chegou certinho, sem nenhum problema!", "positive"),

    # NEGATIVE — 20 avaliações
    ("Meu pedido chegou completamente errado.", "negative"),
    ("A comida estava fria e horrível.", "negative"),
    ("O atendimento foi péssimo.", "negative"),
    ("Meu pedido foi cancelado sem explicação.", "negative"),
    ("A entrega demorou muito!", "negative"),
    ("O lanche chegou todo desmontado.", "negative"),
    ("A embalagem estava aberta.", "negative"),
    ("Faltaram vários itens do meu pedido.", "negative"),
    ("A comida estava estragada.", "negative"),
    ("O entregador foi extremamente grosseiro.", "negative"),
    ("Péssima experiência, não recomendo.", "negative"),
    ("Meu pedido nunca chegou.", "negative"),
    ("O restaurante esqueceu minha bebida.", "negative"),
    ("A comida estava sem sabor e fria.", "negative"),
    ("O pedido veio completamente bagunçado.", "negative"),
    ("Esperei duas horas e nada chegou.", "negative"),
    ("O produto veio diferente da descrição.", "negative"),
    ("A embalagem estava toda danificada.", "negative"),
    ("O atendimento não resolveu meu problema.", "negative"),
    ("Muito decepcionado com a qualidade.", "negative"),

    # NEUTRAL — 20 avaliações
    ("O pedido chegou às 19 horas.", "neutral"),
    ("O restaurante confirmou meu pedido.", "neutral"),
    ("O pedido contém dois hambúrgueres.", "neutral"),
    ("A entrega ocorreu às 20h30.", "neutral"),
    ("Fiz um pedido pelo aplicativo.", "neutral"),
    ("O restaurante fica no centro da cidade.", "neutral"),
    ("Solicitei uma pizza de calabresa.", "neutral"),
    ("O pedido foi registrado no sistema.", "neutral"),
    ("A entrega foi realizada pelo entregador.", "neutral"),
    ("Comprei três refrigerantes.", "neutral"),
    ("O restaurante possui um cardápio digital.", "neutral"),
    ("Meu pedido foi recebido às 18 horas.", "neutral"),
    ("Solicitei um hambúrguer sem cebola.", "neutral"),
    ("O aplicativo informou que saiu para entrega.", "neutral"),
    ("A compra foi realizada pelo celular.", "neutral"),
    ("O pedido possui dois acompanhamentos.", "neutral"),
    ("O restaurante abre às 11 horas.", "neutral"),
    ("Escolhi a opção de pagamento por cartão.", "neutral"),
    ("O pedido foi feito para duas pessoas.", "neutral"),
    ("A entrega foi registrada no aplicativo.", "neutral")
]

# Criando o DataFrame
df_dataset = pd.DataFrame(
    dataset,
    columns=["avaliacao", "sentimento"]
)

# Verificando a distribuição
print("Quantidade total:", len(df_dataset))
print("\nDistribuição das classes:")
print(df_dataset["sentimento"].value_counts())

display(df_dataset.head(10))


Quantidade total: 60

Distribuição das classes:
sentimento
positive    20
negative    20
neutral     20
Name: count, dtype: int64


,avaliacao,sentimento
0,A comida estava deliciosa!,positive
1,A entrega foi excelente e muito rápida!,positive
2,Adorei o atendimento do restaurante!,positive
3,Meu pedido chegou perfeito!,positive
4,O entregador foi muito educado!,positive
5,A embalagem estava ótima e bem organizada!,positive
6,Comida quentinha e muito saborosa!,positive
7,Chegou antes do horário previsto!,positive
8,"Excelente experiência, comprarei novamente!",positive
9,Tudo estava maravilhoso!,positive


In [14]:

df_dev, df_teste = train_test_split(
    df_dataset,
    test_size=0.20,
    random_state=42,
    stratify=df_dataset["sentimento"]
)

print("Desenvolvimento:", len(df_dev))
print("Teste:", len(df_teste))

print("\nDistribuição do conjunto de teste:")
print(df_teste["sentimento"].value_counts())


Desenvolvimento: 48
Teste: 12

Distribuição do conjunto de teste:
sentimento
negative    4
positive    4
neutral     4
Name: count, dtype: int64


In [15]:

from sklearn.metrics import classification_report, confusion_matrix

df_teste = df_teste.copy()

df_teste["previsto"] = [
    analisar_avaliacao(texto)["sentimento"]
    for texto in df_teste["avaliacao"]
]

display(df_teste)

print("\nRelatório de classificação:")

print(classification_report(
    df_teste["sentimento"],
    df_teste["previsto"],
    labels=["negative", "neutral", "positive"],
    zero_division=0
))

print("\nMatriz de confusão:")

print(confusion_matrix(
    df_teste["sentimento"],
    df_teste["previsto"],
    labels=["negative", "neutral", "positive"]
))


,avaliacao,sentimento,previsto
34,O pedido veio completamente bagunçado.,negative,negative
17,O entregador foi atencioso e gentil!,positive,positive
41,O restaurante confirmou meu pedido.,neutral,positive
51,Meu pedido foi recebido às 18 horas.,neutral,neutral
3,Meu pedido chegou perfeito!,positive,positive
26,A embalagem estava aberta.,negative,neutral
8,"Excelente experiência, comprarei novamente!",positive,positive
30,"Péssima experiência, não recomendo.",negative,negative
40,O pedido chegou às 19 horas.,neutral,neutral
57,Escolhi a opção de pagamento por cartão.,neutral,neutral



Relatório de classificação:
              precision    recall  f1-score   support

    negative       1.00      0.75      0.86         4
     neutral       0.75      0.75      0.75         4
    positive       0.80      1.00      0.89         4

    accuracy                           0.83        12
   macro avg       0.85      0.83      0.83        12
weighted avg       0.85      0.83      0.83        12


Matriz de confusão:
[[3 1 0]
 [0 3 1]
 [0 0 4]]


In [ ]:
avaliacoes_erro = [
    "Chegou antes do horário previsto!",
    "O produto veio diferente da descrição."
]

for texto in avaliacoes_erro:

    print("\nAvaliação:", texto)

    resultados = modelo(texto, top_k=None)

    for classe in sorted(
        resultados,
        key=lambda x: x["score"],
        reverse=True
    ):
        print(
            f'{classe["label"]}: '
            f'{classe["score"] * 100:.2f}%'
        )


Avaliação: Chegou antes do horário previsto!
neutral: 49.43%
positive: 47.46%
negative: 3.11%

Avaliação: O produto veio diferente da descrição.
neutral: 53.29%
negative: 43.24%
positive: 3.47%


In [16]:

# Análise das 48 avaliações de desenvolvimento

df_analise = df_dev.copy()

resultados = [
    analisar_avaliacao(texto)
    for texto in df_analise["avaliacao"]
]

df_analise["previsto"] = [
    resultado["sentimento"]
    for resultado in resultados
]

df_analise["confianca"] = [
    resultado["pontuacao"]
    for resultado in resultados
]

df_analise["correto"] = (
    df_analise["sentimento"] == df_analise["previsto"]
)

# Mostrar previsões com menor confiança
df_ordenado = df_analise.sort_values(
    by="confianca",
    ascending=True
)

display(
    df_ordenado[
        ["avaliacao", "sentimento", "previsto",
         "confianca", "correto"]
    ]
)

print("\nQuantidade de erros:")
print((~df_analise["correto"]).sum())

print("\nQuantidade de acertos:")
print(df_analise["correto"].sum())


,avaliacao,sentimento,previsto,confianca,correto
7,Chegou antes do horário previsto!,positive,neutral,0.4943,False
31,Meu pedido nunca chegou.,negative,negative,0.5301,True
36,O produto veio diferente da descrição.,negative,neutral,0.5329,False
35,Esperei duas horas e nada chegou.,negative,negative,0.5943,True
52,Solicitei um hambúrguer sem cebola.,neutral,neutral,0.6228,True
19,"Tudo chegou certinho, sem nenhum problema!",positive,positive,0.6514,True
46,Solicitei uma pizza de calabresa.,neutral,neutral,0.6584,True
49,Comprei três refrigerantes.,neutral,neutral,0.7277,True
53,O aplicativo informou que saiu para entrega.,neutral,neutral,0.7898,True
14,Meu pedido veio completo e bem embalado!,positive,positive,0.7923,True



Quantidade de erros:
2

Quantidade de acertos:
46


In [ ]:

def analisar_avaliacao_detalhada(texto):
    resultados = modelo(texto, top_k=None)

    if resultados and isinstance(resultados[0], list):
        resultados = resultados[0]

    resultados = sorted(
        resultados,
        key=lambda x: x["score"],
        reverse=True
    )

    primeira_classe = resultados[0]
    segunda_classe = resultados[1]

    diferenca = (
        primeira_classe["score"] -
        segunda_classe["score"]
    )

    score_minimo = 0.60
    margem_minima = 0.05

    baixa_pontuacao = (
        primeira_classe["score"] < score_minimo
    )

    margem_pequena = diferenca < margem_minima

    precisa_revisao = (
        baixa_pontuacao or margem_pequena
    )

    return {
        "avaliacao": texto,
        "sentimento": primeira_classe["label"],
        "score_modelo": round(primeira_classe["score"], 4),
        "segunda_classe": segunda_classe["label"],
        "score_segunda_classe": round(segunda_classe["score"], 4),
        "diferenca": round(diferenca, 4),
        "baixa_pontuacao": baixa_pontuacao,
        "margem_pequena": margem_pequena,
        "precisa_revisao": precisa_revisao
    }


Avaliação: Chegou antes do horário previsto!
neutral: 49.43%
positive: 47.46%
negative: 3.11%

Avaliação: O produto veio diferente da descrição.
neutral: 53.29%
negative: 43.24%
positive: 3.47%


In [24]:
avaliacoes_erro = [
    "Chegou antes do horário previsto!",
    "O produto veio diferente da descrição.",
    "A comida estava deliciosa!",
    "Meu pedido nunca chegou."
]

df_detalhado = pd.DataFrame([
    analisar_avaliacao_detalhada(texto)
    for texto in avaliacoes_erro
])

display(df_detalhado)

,avaliacao,sentimento,score_modelo,segunda_classe,score_segunda_classe,diferenca,resultado_ambiguo
0,Chegou antes do horário previsto!,neutral,0.4943,positive,0.4746,0.0198,True
1,O produto veio diferente da descrição.,neutral,0.5329,negative,0.4324,0.1006,False
2,A comida estava deliciosa!,positive,0.9251,neutral,0.0578,0.8673,False
3,Meu pedido nunca chegou.,negative,0.5301,neutral,0.4346,0.0955,False


In [27]:
# Guardar as previsões do primeiro modelo
df_teste = df_teste.copy()

df_teste["previsto_cardiff"] = df_teste["previsto"]

# Liberar o modelo antigo da memória
del modelo

import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Modelo anterior liberado da memória!")

Modelo anterior liberado da memória!


In [28]:
from transformers import pipeline

modelo_2 = pipeline(
    "text-classification",
    model="Horizon-Labs/multilingual-sentiment-small"
)

print("Segundo modelo carregado com sucesso!")

config.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  563MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Segundo modelo carregado com sucesso!


In [29]:
avaliacoes = df_teste["avaliacao"].tolist()

resultados_modelo_2 = modelo_2(avaliacoes)

df_teste["previsto_horizon"] = [
    resultado["label"]
    for resultado in resultados_modelo_2
]

display(
    df_teste[
        [
            "avaliacao",
            "sentimento",
            "previsto_cardiff",
            "previsto_horizon"
        ]
    ]
)

,avaliacao,sentimento,previsto_cardiff,previsto_horizon
34,O pedido veio completamente bagunçado.,negative,negative,negative
17,O entregador foi atencioso e gentil!,positive,positive,positive
41,O restaurante confirmou meu pedido.,neutral,positive,positive
51,Meu pedido foi recebido às 18 horas.,neutral,neutral,neutral
3,Meu pedido chegou perfeito!,positive,positive,positive
26,A embalagem estava aberta.,negative,neutral,neutral
8,"Excelente experiência, comprarei novamente!",positive,positive,positive
30,"Péssima experiência, não recomendo.",negative,negative,negative
40,O pedido chegou às 19 horas.,neutral,neutral,neutral
57,Escolhi a opção de pagamento por cartão.,neutral,neutral,neutral


In [30]:
from sklearn.metrics import classification_report

print("===== CARDIFF =====")

print(classification_report(
    df_teste["sentimento"],
    df_teste["previsto_cardiff"],
    labels=["negative", "neutral", "positive"],
    zero_division=0
))

print("===== HORIZON =====")

print(classification_report(
    df_teste["sentimento"],
    df_teste["previsto_horizon"],
    labels=["negative", "neutral", "positive"],
    zero_division=0
))

===== CARDIFF =====
              precision    recall  f1-score   support

    negative       1.00      0.75      0.86         4
     neutral       0.75      0.75      0.75         4
    positive       0.80      1.00      0.89         4

    accuracy                           0.83        12
   macro avg       0.85      0.83      0.83        12
weighted avg       0.85      0.83      0.83        12

===== HORIZON =====
              precision    recall  f1-score   support

    negative       1.00      0.75      0.86         4
     neutral       0.75      0.75      0.75         4
    positive       0.80      1.00      0.89         4

    accuracy                           0.83        12
   macro avg       0.85      0.83      0.83        12
weighted avg       0.85      0.83      0.83        12



In [31]:
acuracia_cardiff = (
    df_teste["sentimento"] ==
    df_teste["previsto_cardiff"]
).mean()

acuracia_horizon = (
    df_teste["sentimento"] ==
    df_teste["previsto_horizon"]
).mean()

print(f"Cardiff: {acuracia_cardiff:.2%}")
print(f"Horizon: {acuracia_horizon:.2%}")

Cardiff: 83.33%
Horizon: 83.33%


In [32]:
erros_horizon = df_teste[
    df_teste["sentimento"] != df_teste["previsto_horizon"]
]

for texto in erros_horizon["avaliacao"]:

    print("\nAvaliação:", texto)

    resultados = modelo_2(texto, top_k=None)

    if resultados and isinstance(resultados[0], list):
        resultados = resultados[0]

    resultados = sorted(
        resultados,
        key=lambda x: x["score"],
        reverse=True
    )

    for resultado in resultados:
        print(
            f'{resultado["label"]}: '
            f'{resultado["score"] * 100:.2f}%'
        )


Avaliação: O restaurante confirmou meu pedido.
positive: 71.81%
neutral: 27.42%
negative: 0.77%

Avaliação: A embalagem estava aberta.
neutral: 92.46%
positive: 6.89%
negative: 0.65%


In [33]:
concordancia = (
    df_teste["previsto_cardiff"] ==
    df_teste["previsto_horizon"]
).mean()

print(f"Concordância entre modelos: {concordancia:.2%}")

Concordância entre modelos: 100.00%


In [34]:
import requests

url = "https://data.mendeley.com/public-files/datasets/t5cnf8krvp/files"

resposta = requests.get(url)

print("Status HTTP:", resposta.status_code)
print("Tipo de conteúdo:", resposta.headers.get("Content-Type"))
print("Tamanho da resposta:", len(resposta.content))

Status HTTP: 404
Tipo de conteúdo: text/html; charset=utf-8
Tamanho da resposta: 177


In [35]:
import requests

url = "https://api.data.mendeley.com/datasets/t5cnf8krvp"

params = {
    "version": 2
}

resposta = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status HTTP:", resposta.status_code)
print("URL consultada:", resposta.url)

if resposta.status_code == 200:
    dados = resposta.json()

    print("\nNome:", dados.get("name"))
    print("Versão:", dados.get("version"))
    print("Descrição:", dados.get("description"))
else:
    print("\nErro:", resposta.text[:500])

Status HTTP: 401
URL consultada: https://api.data.mendeley.com/datasets/t5cnf8krvp?version=2

Erro: {"message":"Unauthorized (no suitable authorization credentials were provided)"}


In [36]:
import requests

url = (
    "https://api.data.mendeley.com/"
    "datasets/publics/t5cnf8krvp/files"
)

resposta = requests.get(
    url,
    params={"version": 2},
    timeout=30
)

print("Status HTTP:", resposta.status_code)
print("Content-Type:", resposta.headers.get("Content-Type"))

if resposta.status_code == 200:
    arquivos = resposta.json()

    for arquivo in arquivos:
        print("\nNome:", arquivo.get("filename"))
        print("ID:", arquivo.get("id"))
        print("Tamanho:", arquivo.get("size"))
else:
    print("Resposta:", resposta.text[:500])

Status HTTP: 401
Content-Type: application/json; charset=utf-8
Resposta: {"message":"Unauthorized (no suitable authorization credentials were provided)"}


In [37]:
from google.colab import files

arquivos_enviados = files.upload()

print("Arquivos recebidos:")

for nome in arquivos_enviados:
    print(nome)

Saving Value Co-Destruction in Food Delivery Apps”.zip to Value Co-Destruction in Food Delivery Apps”.zip
Arquivos recebidos:
Value Co-Destruction in Food Delivery Apps”.zip


In [38]:
import zipfile
import os

arquivo_zip = next(
    nome for nome in arquivos_enviados
    if nome.endswith(".zip")
)

with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
    zip_ref.extractall("/content/dataset_delivery")

print("Arquivos extraídos:")

for raiz, diretorios, arquivos in os.walk("/content/dataset_delivery"):
    for arquivo in arquivos:
        print(os.path.join(raiz, arquivo))

Arquivos extraídos:
/content/dataset_delivery/Value Co-Destruction in Food Delivery Apps”/artigo_google_play.py


In [39]:
caminho = (
    "/content/dataset_delivery/"
    "Value Co-Destruction in Food Delivery Apps”/"
    "artigo_google_play.py"
)

with open(caminho, "r", encoding="utf-8") as arquivo:
    codigo = arquivo.read()

print(codigo[:12000])

# -*- coding: utf-8 -*-
"""Artigo Google Play.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/19slsI3SMG6q3RNc-YLY2vb6YGgdYtlIw
"""

import nltk
nltk.download('vader_lexicon')

pip install google-play-scraper

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google_play_scraper import app, reviews
import numpy as np

# Configurar estilo
plt.style.use('ggplot')
sns.set_palette("viridis")
plt.rcParams['font.size'] = 12

# Lista de apps para análise
app_ids = [ 'br.com.brainweb.ifood',
    'com.cerveceriamodelo.modelonow',
    'com.mcdo.mcdonalds',
    'habibs.alphacode.com.br',
    'com.daki',
    'com.foodtosave.foodtosave',
    'com.grability.rappi',
    'burgerking.com.br.appandroid',
    'com.cocobambu.cupdescprod',
    'com.vanuatu.aiqfome']

# Função de raspagem corrigida
def scrape_reviews(app_id):
    try:
        app_info = app(app_id, lang='pt-BR')
        result, _ = reviews(
      

In [40]:
!pip -q install google-play-scraper

In [41]:
import pandas as pd
from google_play_scraper import app, reviews, Sort

app_id = "br.com.brainweb.ifood"

# Obter informações do aplicativo
informacoes = app(
    app_id,
    lang="pt",
    country="br"
)

# Coletar até 100 avaliações mais recentes
avaliacoes, token = reviews(
    app_id,
    lang="pt",
    country="br",
    sort=Sort.NEWEST,
    count=100
)

# Selecionar somente os campos necessários
dados = []

for avaliacao in avaliacoes:
    texto = avaliacao.get("content")

    if texto:
        dados.append({
            "app": informacoes["title"],
            "app_id": app_id,
            "review_date": avaliacao["at"],
            "rating": avaliacao["score"],
            "review_text": texto
        })

df_delivery = pd.DataFrame(dados)

print("Aplicativo:", informacoes["title"])
print("Avaliações coletadas:", len(df_delivery))

display(df_delivery.head(10))

Aplicativo: iFood comida e mercado em casa
Avaliações coletadas: 100


,app,app_id,review_date,rating,review_text
0,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:56:20,5,app muito bom
1,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:55:40,5,bom de mais
2,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:51:08,5,muito rápido e fácil
3,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:44:58,1,"O aplicativo está horrível, simplesmente não d..."
4,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:40:01,1,Sistema horrível.
5,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:37:34,5,muito bom desconto ótimo
6,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:32:14,5,ifood é nosso
7,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:28:44,5,top dms
8,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:27:48,5,cumpre o que promete
9,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:23:32,5,ótima plataforma para comprar no geral desde c...


In [42]:
print("Distribuição das avaliações:")

print(
    df_delivery["rating"]
    .value_counts()
    .sort_index()
)

print("\nInformações do dataset:")

df_delivery.info()

Distribuição das avaliações:
rating
1    27
2     4
4     4
5    65
Name: count, dtype: int64

Informações do dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   app          100 non-null    object        
 1   app_id       100 non-null    object        
 2   review_date  100 non-null    datetime64[ns]
 3   rating       100 non-null    int64         
 4   review_text  100 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 4.0+ KB


In [43]:
def classificar_sentimento(nota):

    if nota <= 2:
        return "negative"

    elif nota == 3:
        return "neutral"

    else:
        return "positive"


df_delivery["sentiment"] = (
    df_delivery["rating"]
    .apply(classificar_sentimento)
)

display(
    df_delivery[
        ["review_text", "rating", "sentiment"]
    ].head(10)
)

,review_text,rating,sentiment
0,app muito bom,5,positive
1,bom de mais,5,positive
2,muito rápido e fácil,5,positive
3,"O aplicativo está horrível, simplesmente não d...",1,negative
4,Sistema horrível.,1,negative
5,muito bom desconto ótimo,5,positive
6,ifood é nosso,5,positive
7,top dms,5,positive
8,cumpre o que promete,5,positive
9,ótima plataforma para comprar no geral desde c...,5,positive


In [47]:
# Procurar células relacionadas ao Qwen e à inferência

keywords = [
    "from_pretrained",
    "AutoTokenizer",
    "AutoModel",
    "prompt",
    "messages",
    "generate",
    "pipeline"
]

for i, cell in enumerate(get_ipython().user_ns.get(
    "In", []
)):
    if any(word in cell for word in keywords):
        print(f"\n{'='*50}")
        print(f"CÉLULA {i}")
        print('='*50)
        print(cell[:3000])


CÉLULA 7
from transformers import pipeline

print("Transformers importado com sucesso!")

CÉLULA 8
from transformers import pipeline

modelo = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

print("Modelo carregado com sucesso!")

CÉLULA 28
from transformers import pipeline

modelo_2 = pipeline(
    "text-classification",
    model="Horizon-Labs/multilingual-sentiment-small"
)

print("Segundo modelo carregado com sucesso!")

CÉLULA 47
# Procurar células relacionadas ao Qwen e à inferência

keywords = [
    "from_pretrained",
    "AutoTokenizer",
    "AutoModel",
    "prompt",
    "messages",
    "generate",
    "pipeline"
]

for i, cell in enumerate(get_ipython().user_ns.get(
    "In", []
)):
    if any(word in cell for word in keywords):
        print(f"\n{'='*50}")
        print(f"CÉLULA {i}")
        print('='*50)
        print(cell[:3000])


In [44]:
# Verificar a distribuição dos sentimentos
print("Distribuição dos sentimentos:")
print(df['sentiment'].value_counts())

# Verificar possíveis valores ausentes
print("\nValores ausentes:")
print(df.isnull().sum())

# Verificar avaliações duplicadas
print("\nAvaliações duplicadas:")
print(df['review_text'].duplicated().sum())

# Visualizar algumas avaliações
display(df[['review_text', 'rating', 'sentiment']].sample(10))

Distribuição dos sentimentos:


KeyError: 'sentiment'

In [45]:
# Verificar as colunas existentes no DataFrame
print("Colunas disponíveis:")
print(df.columns.tolist())

# Verificar as primeiras linhas
display(df.head())

Colunas disponíveis:
['avaliacao', 'sentimento', 'pontuacao']


,avaliacao,sentimento,pontuacao
0,A entrega foi excelente!,positive,0.9150
1,Meu pedido chegou completamente errado.,negative,0.8888
2,O atendimento foi razoável.,positive,0.8876


In [46]:
print("Distribuição dos sentimentos:")
print(df['sentimento'].value_counts())

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nAvaliações duplicadas:")
print(df['avaliacao'].duplicated().sum())

print("\nVisualização das avaliações:")
display(df.sample(min(10, len(df))))

Distribuição dos sentimentos:
sentimento
positive    2
negative    1
Name: count, dtype: int64

Valores ausentes:
avaliacao     0
sentimento    0
pontuacao     0
dtype: int64

Avaliações duplicadas:
0

Visualização das avaliações:


,avaliacao,sentimento,pontuacao
2,O atendimento foi razoável.,positive,0.8876
0,A entrega foi excelente!,positive,0.9150
1,Meu pedido chegou completamente errado.,negative,0.8888


In [48]:
# Listar todas as células executadas no notebook

for i, cell in enumerate(get_ipython().user_ns.get("In", [])):
    if cell.strip():
        print(f"\n{'='*60}")
        print(f"CÉLULA {i}")
        print('='*60)
        print(cell[:1500])


CÉLULA 1
print("Olá! Minha primeira inteligência artificial está começando!")

CÉLULA 2
get_ipython().system('pip install -U huggingface_hub')

CÉLULA 3
get_ipython().system('pip install -U "huggingface_hub<2.0"')

CÉLULA 4
import huggingface_hub
from huggingface_hub import HfApi

print("Hugging Face instalado!")
print("Versão:", huggingface_hub.__version__)

api = HfApi()

print("API inicializada com sucesso!")

CÉLULA 5
import huggingface_hub
from huggingface_hub import HfApi

print("Biblioteca:", huggingface_hub.__version__)

api = HfApi()

print("Conexão com a API inicializada!")

CÉLULA 6
get_ipython().system('pip install -q transformers torch')

CÉLULA 7
from transformers import pipeline

print("Transformers importado com sucesso!")

CÉLULA 8
from transformers import pipeline

modelo = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

print("Modelo carregado com sucesso!")

CÉLULA 9
avaliacoes = [
    "A comida chegou muito rápido e

In [50]:
import pandas as pd
import os

# Criar uma cópia dos dados originais
df_raw = df_delivery[
    [
        "app",
        "app_id",
        "review_date",
        "rating",
        "review_text"
    ]
].copy()

# Criar uma versão processada
df_processed = df_raw.copy()

# Criar rótulos baseados nas avaliações
def classificar_sentimento(nota):

    if nota <= 2:
        return "negative"

    elif nota == 3:
        return "neutral"

    else:
        return "positive"


df_processed["sentiment"] = (
    df_processed["rating"]
    .apply(classificar_sentimento)
)

# Criar diretórios
os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/data/processed", exist_ok=True)

# Salvar os arquivos
df_raw.to_csv(
    "/content/data/raw/reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

df_processed.to_csv(
    "/content/data/processed/reviews_labeled.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos criados com sucesso!")

print("\nDataset original:")
display(df_raw.head())

print("\nDataset processado:")
display(df_processed.head())

print("\nDistribuição dos sentimentos:")
print(df_processed["sentiment"].value_counts())

Arquivos criados com sucesso!

Dataset original:


,app,app_id,review_date,rating,review_text
0,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:56:20,5,app muito bom
1,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:55:40,5,bom de mais
2,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:51:08,5,muito rápido e fácil
3,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:44:58,1,"O aplicativo está horrível, simplesmente não d..."
4,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:40:01,1,Sistema horrível.



Dataset processado:


,app,app_id,review_date,rating,review_text,sentiment
0,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:56:20,5,app muito bom,positive
1,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:55:40,5,bom de mais,positive
2,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:51:08,5,muito rápido e fácil,positive
3,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:44:58,1,"O aplicativo está horrível, simplesmente não d...",negative
4,iFood comida e mercado em casa,br.com.brainweb.ifood,2026-11-03 18:40:01,1,Sistema horrível.,negative



Distribuição dos sentimentos:
sentiment
positive    69
negative    31
Name: count, dtype: int64


In [51]:
from google.colab import files

files.download("/content/data/raw/reviews.csv")
files.download("/content/data/processed/reviews_labeled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>